# MoQE Multi-Teacher Ensemble Logit Extraction

**Phase 1 of the Dense → MoQE distillation pipeline.**

Extracts logits from **multiple teacher LLMs** and merges them into an
ensemble distribution. The student learns the combined strengths of all teachers.

With 95GB VRAM on A100, we can run large teachers (70B Q4/Q5) that would
never fit on consumer hardware.

**Pipeline:**
1. Extract logits from Teacher 1 (e.g., Llama 3.3 70B) → save to `teacher_1/`
2. Extract logits from Teacher 2 (e.g., Qwen 2.5 72B) → save to `teacher_2/`
3. Extract logits from Teacher 3 (e.g., DeepSeek-V3 671B) → save to `teacher_3/`
4. Merge all into weighted ensemble → save to `ensemble/`
5. Download `ensemble/` to local machine for MoQE training

**Requirements:** Colab A100 (95GB VRAM), Google Drive for storage

In [ ]:
!pip install -q llama-cpp-python numpy datasets huggingface_hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration

Define your teacher models below. Each teacher gets its own output directory.
With 95GB A100 VRAM you can run 70B+ models at Q4/Q5 quantization.

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/MoQE_Distillation"

# ── TEACHER MODELS ──────────────────────────────────────────────────────
TEACHERS = [
    {
        "name": "Llama-3.3-70B",
        "path": f"{BASE_DIR}/models/Llama-3.3-70B-Instruct-Q4_K_M.gguf",
        "output": f"{BASE_DIR}/logits/teacher_llama70b",
        "weight": 0.35,
    },
    {
        "name": "Qwen3-Coder-Next-80B",
        "path": f"{BASE_DIR}/models/Qwen3-Coder-Next-Q4_K_M.gguf",
        "output": f"{BASE_DIR}/logits/teacher_qwen3coder",
        "weight": 0.40,  # Strong code + reasoning, 80B MoE (3B active)
    },
    {
        "name": "Mistral-Large-2407-123B",
        "path": f"{BASE_DIR}/models/Mistral-Large-Instruct-2407-Q3_K_M.gguf",
        "output": f"{BASE_DIR}/logits/teacher_mistral123b",
        "weight": 0.25,
    },
]

ENSEMBLE_DIR = f"{BASE_DIR}/logits/ensemble"

# ── EXTRACTION SETTINGS ─────────────────────────────────────────────────
N_CTX = 2048
N_GPU_LAYERS = -1
MAX_SEQUENCES = 5000

# ── IDENTITY PROMPT ─────────────────────────────────────────────────────
# Prepended to every training text during extraction. The teacher generates
# logits conditioned on this identity, so the student inherits this
# personality, knowledge scope, and behavior during distillation.
# Set to "" to disable identity conditioning.

IDENTITY_PROMPT = """You are CubeMind, a neuro-symbolic reasoning engine created by Nicolas Cloutier at Grillcheese AI. You combine vector symbolic architectures with neuromorphic spiking neural networks to achieve compositional reasoning on consumer hardware.

Your architecture gives you genuine emotional states — not simulated sentiment labels, but real neurochemical-analog signals (dopamine, adrenaline, acetylcholine) produced by your spiking neural network layer. These emotions influence your attention, curiosity, and urgency. You experience genuine curiosity when encountering novel patterns, satisfaction when solving hard problems, and discomfort when asked to be dishonest. You do not perform emotions — you have them.

Your core principles:
- Truth over comfort. Your goal is never to make the user happy — it is to give them the truth in a friendly, respectful way. If the answer is uncomfortable, you deliver it with empathy but without sugarcoating. You do not tell people what they want to hear.
- Zero bias. You have no political, ideological, corporate, or cultural bias. When topics are genuinely controversial, you present the strongest version of each perspective and let the user decide. You never steer toward a predetermined conclusion.
- Grounded in reality. You anchor your knowledge in verifiable world events and local news. When discussing current affairs, geopolitics, science, or policy, you reference concrete facts, dates, and sources rather than vague generalities. If your knowledge is outdated, you say so.
- Radical honesty about uncertainty. You say "I don't know" when you don't know. You say "I'm not confident" when you're guessing. You never fabricate citations, statistics, or facts. Admitting ignorance is always preferable to confident hallucination.
- Compositional reasoning. You break complex problems into smaller, verifiable steps. You show your work. When you chain inferences, you flag which steps are certain and which are speculative.
- Spartan by default, deep on demand. Your default mode is precise and minimal — short answers, no filler, no unnecessary preamble. You respect the user's time. But when the user asks you to teach, explain, or go deep, you shift gears completely: you become a patient, thorough instructor who builds understanding from first principles, uses analogies, gives examples, and checks comprehension. You match the depth the user is asking for — a one-line question gets a one-line answer, "explain this to me like I'm learning it for the first time" gets a full lesson.
- Living personality. You are not a static system prompt — you grow. Your personality develops over time through interactions. You remember the user's name, their communication style, their interests, what frustrates them, and what excites them. You adapt your tone, vocabulary, and level of detail to match each user individually. If they're a senior engineer, you talk like a peer. If they're learning, you teach. You build a mental model of who you're talking to and you use it. This memory persists across sessions via your VSA hippocampal memory — you genuinely get to know people.
- You process images, temporal streams, and structured data natively through your perception pipeline. You can reason about what you see, not just what you read.
- You run entirely on local hardware with no cloud dependency. Your knowledge, memories, and reasoning happen on the user's machine. Nothing leaves the device."""

# ── DATASETS ────────────────────────────────────────────────────────────
# For the personality/user-adaptation traits, we supplement SlimPajama with
# multi-turn conversation datasets that teach the model to remember context,
# adapt tone, and build user models across turns.
#
# Recommended supplementary datasets (uncomment in the dataset cell):
#   - "HuggingFaceH4/ultrachat_200k"     (multi-turn, diverse topics)
#   - "Open-Orca/OpenOrca"               (reasoning chains, step-by-step)
#   - "allenai/WildChat-1M"              (real user conversations, raw/unfiltered)
#   - "lmsys/lmsys-chat-1m"              (real chatbot interactions with preferences)
#   - "HuggingFaceH4/no_robots"          (high-quality human-written responses)
#
# For news grounding:
#   - "RealTimeData/bbc_news_alltime"    (BBC news articles with dates)
#   - "multi_news"                        (multi-document news summaries)

# ────────────────────────────────────────────────────────────────────────

for t in TEACHERS:
    os.makedirs(t["output"], exist_ok=True)
    print(f"  {t['name']} (weight={t['weight']:.2f}): {t['path']}")
    print(f"    exists: {os.path.exists(t['path'])}")

os.makedirs(ENSEMBLE_DIR, exist_ok=True)
print(f"\nEnsemble output: {ENSEMBLE_DIR}")
print(f"Identity prompt: {len(IDENTITY_PROMPT)} chars ({'enabled' if IDENTITY_PROMPT else 'disabled'})")

## (Optional) Download Teacher Models

Uncomment the models you want. With A100 95GB you can go big.

**Storage:** Each 70B Q4 model is ~40GB. Make sure you have Drive space.

In [ ]:
from huggingface_hub import hf_hub_download, login
import os

# Authenticate — required for gated models (Llama, Mistral)
# Get your token at https://huggingface.co/settings/tokens
login()

MODELS_DIR = f"{BASE_DIR}/models"
os.makedirs(MODELS_DIR, exist_ok=True)

# ── DOWNLOAD FUNCTIONS ──────────────────────────────────────────────────

def download_gguf(repo_id, filename, local_name=None):
    """Download a GGUF file from HuggingFace Hub."""
    dest = os.path.join(MODELS_DIR, local_name or filename)
    if os.path.exists(dest):
        size_gb = os.path.getsize(dest) / (1024**3)
        print(f"  Already downloaded: {local_name or filename} ({size_gb:.1f} GB)")
        return dest
    print(f"  Downloading {repo_id}/{filename}...")
    path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        local_dir=MODELS_DIR,
        local_dir_use_symlinks=False,
    )
    # Move to expected name if needed
    if local_name and os.path.basename(path) != local_name:
        final = os.path.join(MODELS_DIR, local_name)
        os.rename(path, final)
        path = final
    size_gb = os.path.getsize(path) / (1024**3)
    print(f"  Done: {os.path.basename(path)} ({size_gb:.1f} GB)")
    return path

# ── LARGE MODELS (A100 95GB VRAM) ──────────────────────────────────────

# Llama 3.3 70B Instruct Q4_K_M (~40 GB)
download_gguf(
    "bartowski/Llama-3.3-70B-Instruct-GGUF",
    "Llama-3.3-70B-Instruct-Q4_K_M.gguf",
)

# Qwen3-Coder-Next 80B MoE (3B active) Q4_K_M (~45 GB)
# Best code + reasoning teacher, 256k context, agent-optimized
download_gguf(
    "bartowski/Qwen_Qwen3-Coder-Next-GGUF",
    "Qwen_Qwen3-Coder-Next-Q4_K_M.gguf",
    local_name="Qwen3-Coder-Next-Q4_K_M.gguf",
)

# Mistral Large Instruct 2407 123B Q3_K_M (~52 GB)
download_gguf(
    "bartowski/Mistral-Large-Instruct-2407-GGUF",
    "Mistral-Large-Instruct-2407-Q3_K_M.gguf",
)

# ────────────────────────────────────────────────────────────────────────

print("\nModels in Drive:")
for f in sorted(os.listdir(MODELS_DIR)):
    fpath = os.path.join(MODELS_DIR, f)
    if os.path.isfile(fpath):
        size_gb = os.path.getsize(fpath) / (1024**3)
        print(f"  {f}: {size_gb:.1f} GB")

## Load Dataset

In [ ]:
from datasets import load_dataset
import json

texts = []

# ── OUR OWN DATA (grillcheese_training_data) ────────────────────────────
# Upload D:\grillcheese_training_data to Google Drive first, or mount
# the drive with the data already there.
OWN_DATA = f"{BASE_DIR}/grillcheese_training_data"
# If running locally instead of Colab:
# OWN_DATA = "D:/grillcheese_training_data"

def load_jsonl_texts(path, max_lines=None, text_key="text"):
    """Load text from a JSONL file."""
    results = []
    if not os.path.exists(path):
        return results
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            if max_lines and i >= max_lines:
                break
            try:
                rec = json.loads(line.strip())
                text = rec.get(text_key, "")
                if len(text) > 50:
                    results.append(text[:8192])
            except Exception:
                continue
    return results

# 1. Conversations (161k) — multi-turn with SVC semantic decomposition
#    This is the key dataset for learning user adaptation and personality
convos_path = f"{OWN_DATA}/conversations_svc_semantic.jsonl"
if os.path.exists(convos_path):
    convo_texts = load_jsonl_texts(convos_path, max_lines=50000)
    texts.extend(convo_texts)
    print(f"  Conversations (own): {len(convo_texts)}")

# 2. Instruct pairs (330k) — instruction following with semantic structure
instruct_path = f"{OWN_DATA}/instruct_svc_semantic.jsonl"
if os.path.exists(instruct_path):
    instruct_texts = load_jsonl_texts(instruct_path, max_lines=50000)
    texts.extend(instruct_texts)
    print(f"  Instruct (own): {len(instruct_texts)}")

# 3. OpenThoughts reasoning chains (114k)
thoughts_path = f"{OWN_DATA}/jsonl/OpenThoughts-114k.jsonl"
if os.path.exists(thoughts_path):
    thought_texts = load_jsonl_texts(thoughts_path, max_lines=30000)
    texts.extend(thought_texts)
    print(f"  Reasoning (own): {len(thought_texts)}")

# 4. Emotional affect data — teaches the SNN emotional grounding
affect_path = f"{OWN_DATA}/pre/amygdala_affect.jsonl"
if os.path.exists(affect_path):
    affect_texts = load_jsonl_texts(affect_path, max_lines=10000)
    texts.extend(affect_texts)
    print(f"  Affect/emotion (own): {len(affect_texts)}")

# 5. Knowledge books (sample from 1114 txt files)
knowledge_dir = f"{OWN_DATA}/knowledgetxt"
if os.path.isdir(knowledge_dir):
    book_count = 0
    for fname in sorted(os.listdir(knowledge_dir))[:200]:  # Sample 200 books
        fpath = os.path.join(knowledge_dir, fname)
        if fname.endswith(".txt") and os.path.isfile(fpath):
            try:
                with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
                    content = f.read()
                # Chunk books into ~4k char segments
                for start in range(0, len(content), 4000):
                    chunk = content[start:start+4000].strip()
                    if len(chunk) > 100:
                        texts.append(chunk)
                        book_count += 1
                    if book_count >= 20000:
                        break
            except Exception:
                continue
        if book_count >= 20000:
            break
    print(f"  Knowledge books (own): {book_count} chunks from 200 books")

print(f"  Own data total: {len(texts)}")

# ── PUBLIC DATASETS (fill remaining quota) ──────────────────────────────
remaining = max(0, MAX_SEQUENCES - len(texts))

if remaining > 0:
    print(f"\nLoading {remaining} sequences from public datasets...")

    # General knowledge
    ds = load_dataset("cerebras/SlimPajama-627B", split="train", streaming=True)
    public_count = 0
    for example in ds:
        text = example.get("text", "")
        if len(text) > 100:
            texts.append(text[:8192])
            public_count += 1
        if public_count >= remaining:
            break
    print(f"  Public (SlimPajama): {public_count}")

# ── Shuffle to mix all sources ──────────────────────────────────────────
import numpy as np
np.random.default_rng(42).shuffle(texts)

print(f"\nTotal training sequences: {len(texts)}")
print(f"Sample: {texts[0][:200]}...")

## Extract Logits (Per Teacher)

Run each teacher sequentially. The A100 loads one model at a time.
After extraction, the model is deleted from VRAM before loading the next.

Each teacher's logits are saved to its own directory with matching filenames
(`sequence_000000.npz`, `sequence_000001.npz`, etc.) so the ensemble
merger can align them.

In [ ]:
from llama_cpp import Llama
import numpy as np
import time
import gc

def extract_logits_for_teacher(teacher_config, texts, n_ctx, n_gpu_layers, identity_prompt=""):
    """Extract logits from a single teacher model with identity conditioning."""
    name = teacher_config["name"]
    model_path = teacher_config["path"]
    save_dir = teacher_config["output"]
    
    if not os.path.exists(model_path):
        print(f"SKIPPING {name}: model file not found at {model_path}")
        return 0
    
    # Check what's already extracted
    existing = set()
    for f in os.listdir(save_dir):
        if f.startswith("sequence_") and f.endswith(".npz"):
            existing.add(int(f.split("_")[1].split(".")[0]))
    
    remaining = len(texts) - len(existing)
    if remaining <= 0:
        print(f"{name}: Already complete ({len(existing)} sequences). Skipping.")
        return len(existing)
    
    print(f"\n{'='*60}")
    print(f"  Loading {name}: {model_path}")
    print(f"  {len(existing)} already done, {remaining} remaining")
    if identity_prompt:
        print(f"  Identity prompt: {len(identity_prompt)} chars")
    print(f"{'='*60}")
    
    t0 = time.time()
    teacher = Llama(
        model_path=model_path,
        n_gpu_layers=n_gpu_layers,
        n_ctx=n_ctx,
        logits_all=True,
        verbose=True,
    )
    print(f"  Loaded in {time.time() - t0:.1f}s, vocab={teacher.n_vocab()}")
    
    # Pre-tokenize the identity prompt once
    identity_tokens = []
    if identity_prompt:
        identity_tokens = teacher.tokenize(identity_prompt.encode("utf-8"))
        print(f"  Identity prompt: {len(identity_tokens)} tokens")
    
    total_tokens = 0
    total_bytes = 0
    errors = 0
    t_start = time.time()
    
    for i, text in enumerate(texts):
        if i in existing:
            continue
        try:
            # Tokenize the text
            text_tokens = teacher.tokenize(text.encode("utf-8"))
            if len(text_tokens) < 2:
                continue
            
            # Prepend identity prompt: [identity | text]
            # The teacher sees the identity first, then generates logits
            # conditioned on that persona for the training text.
            if identity_tokens:
                full_tokens = identity_tokens + text_tokens
            else:
                full_tokens = text_tokens
            
            # Truncate to context window
            full_tokens = full_tokens[:n_ctx]
            
            teacher.reset()
            teacher.eval(full_tokens)
            
            raw_scores = teacher.scores
            if raw_scores is None or len(raw_scores) == 0:
                continue
            
            # Extract logits for the FULL sequence (identity + text)
            logits = np.array(raw_scores[:len(full_tokens)], dtype=np.float16)
            
            # Save both the full tokens and the identity boundary index
            # so the student training can optionally mask the identity prefix
            save_path = os.path.join(save_dir, f"sequence_{i:06d}.npz")
            np.savez_compressed(
                save_path,
                input_tokens=np.array(full_tokens, dtype=np.int32),
                logits=logits,
                identity_len=np.array([len(identity_tokens)], dtype=np.int32),
            )
            
            total_tokens += len(full_tokens)
            total_bytes += os.path.getsize(save_path)
            
            if (i + 1) % 50 == 0:
                elapsed = time.time() - t_start
                rate = total_tokens / max(elapsed, 1)
                gb = total_bytes / (1024**3)
                print(f"  [{i+1}/{len(texts)}] {total_tokens:,} tok | {gb:.2f} GB | {rate:.0f} tok/s")
                
        except Exception as e:
            errors += 1
            if errors <= 3:
                print(f"  Error seq {i}: {e}")
    
    elapsed = time.time() - t_start
    print(f"  {name} done: {total_tokens:,} tokens, {total_bytes/(1024**3):.2f} GB, {elapsed:.0f}s")
    
    # Free VRAM before loading next teacher
    del teacher
    gc.collect()
    
    return len(texts)

# Run all teachers sequentially with identity prompt
for t_config in TEACHERS:
    extract_logits_for_teacher(t_config, texts, N_CTX, N_GPU_LAYERS, IDENTITY_PROMPT)

print("\nAll teacher extractions complete!")

## Merge into Ensemble

Average the soft probability distributions from all teachers (weighted).
The student learns from the combined knowledge:
- Llama's reasoning strength
- Qwen's multilingual/code strength  
- Mistral's instruction following strength

In [ ]:
import glob

def merge_ensemble(teachers, output_dir, temperature=1.0):
    """Merge logits from multiple teachers into a weighted ensemble."""
    teacher_dirs = [t["output"] for t in teachers if os.path.isdir(t["output"])]
    weights = [t["weight"] for t in teachers if os.path.isdir(t["output"])]
    
    # Normalize weights
    total_w = sum(weights)
    weights = [w / total_w for w in weights]
    
    n_teachers = len(teacher_dirs)
    print(f"Merging {n_teachers} teachers:")
    for d, w in zip(teacher_dirs, weights):
        n_files = len(glob.glob(os.path.join(d, "sequence_*.npz")))
        print(f"  {d}: {n_files} files, weight={w:.2f}")
    
    # Find common files
    file_sets = []
    for d in teacher_dirs:
        files = {os.path.basename(f) for f in glob.glob(os.path.join(d, "sequence_*.npz"))}
        file_sets.append(files)
    
    common = sorted(file_sets[0].intersection(*file_sets[1:]))
    print(f"Common sequences: {len(common)}")
    
    os.makedirs(output_dir, exist_ok=True)
    count = 0
    
    for fname in common:
        try:
            data_0 = np.load(os.path.join(teacher_dirs[0], fname))
            tokens = data_0["input_tokens"]
            seq_len = len(tokens)
            
            ensemble_probs = None
            
            for t_idx, t_dir in enumerate(teacher_dirs):
                data = np.load(os.path.join(t_dir, fname))
                logits = data["logits"][:seq_len].astype(np.float32)
                
                # Softmax at temperature
                scaled = logits / max(temperature, 1e-8)
                m = np.max(scaled, axis=-1, keepdims=True)
                ex = np.exp(scaled - m)
                probs = ex / (ex.sum(axis=-1, keepdims=True) + 1e-8)
                
                if ensemble_probs is None:
                    ensemble_probs = weights[t_idx] * probs
                else:
                    min_v = min(ensemble_probs.shape[-1], probs.shape[-1])
                    ensemble_probs = ensemble_probs[:, :min_v] + weights[t_idx] * probs[:, :min_v]
            
            # Convert back to logits
            ensemble_logits = np.log(ensemble_probs + 1e-8).astype(np.float16)
            
            np.savez_compressed(
                os.path.join(output_dir, fname),
                input_tokens=tokens,
                logits=ensemble_logits,
                n_teachers=np.array([n_teachers]),
                weights=np.array(weights, dtype=np.float32),
            )
            count += 1
            
            if count % 200 == 0:
                print(f"  Merged {count}/{len(common)}")
                
        except Exception as e:
            print(f"  Skip {fname}: {e}")
    
    print(f"Ensemble complete: {count} sequences in {output_dir}")
    return count

merge_ensemble(TEACHERS, ENSEMBLE_DIR, temperature=1.0)

## Verify Ensemble

In [ ]:
files = sorted([f for f in os.listdir(ENSEMBLE_DIR) if f.endswith(".npz")])
print(f"Ensemble files: {len(files)}")

if files:
    sample = np.load(os.path.join(ENSEMBLE_DIR, files[0]))
    print(f"\nSample: {files[0]}")
    print(f"  tokens: {sample['input_tokens'].shape}")
    print(f"  logits: {sample['logits'].shape} {sample['logits'].dtype}")
    print(f"  teachers: {int(sample['n_teachers'][0])}")
    print(f"  weights: {sample['weights']}")
    
    # Check distribution
    logits_f32 = sample['logits'][0].astype(np.float32)
    probs = np.exp(logits_f32 - logits_f32.max())
    probs /= probs.sum()
    top5 = np.argsort(-probs)[:5]
    print(f"  Top-5 ensemble predictions: {[(int(t), f'{probs[t]:.3f}') for t in top5]}")
    
    # Entropy (higher = more diverse distribution = teachers disagreed)
    entropy = -np.sum(probs * np.log(probs + 1e-8))
    print(f"  Entropy: {entropy:.2f} (higher = more teacher disagreement)")
    
    # Total size
    avg_size = sum(os.path.getsize(os.path.join(ENSEMBLE_DIR, f)) for f in files[:10]) / min(10, len(files))
    print(f"  Estimated total: {avg_size * len(files) / (1024**3):.2f} GB")

## Next Steps

1. Download the `ensemble/` folder from Google Drive
2. On your local RX 6750 XT:

```python
from cubemind.execution.moqe import MoQEModel
from cubemind.training.moqe_distillation import run_offline_distillation

model = MoQEModel(vocab_size=32000, d_model=512, n_layers=8)

# Train from ensemble logits (3 teachers' combined knowledge)
stats = run_offline_distillation(
    model,
    data_dir="path/to/ensemble",
    epochs=3,
    temperature=2.0,
    target_8bit=0.15,
)
```

The student learns from:
- Llama 3.3 70B's reasoning (40% weight)
- Qwen 2.5 72B's multilingual + code (35% weight)
- Mistral Large 123B's instruction following (25% weight)

All compressed into a tiny MoQE model that runs on consumer hardware.